# Matrices

## What's covered

- Two views of a matrix — a list of column vectors, or a linear transformation
- **Matrix-vector multiplication** `Ax` — as a linear combination of columns
- **Matrix-matrix multiplication** `AB` — composition of transformations; why order matters
- **Transpose**, **identity**, and **inverse**
- Special matrices — diagonal, symmetric, orthogonal — and what makes each useful
- **Determinant** — the area / volume scaling factor of a transformation
- Where this appears in ML — neural net layers, change of basis, rotations, attention

Canonical matrix used throughout:

$$
A = \begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix},
\qquad
\det(A) = 5
$$


## Two views of a matrix

Just like a vector has two heads (arrow + list), a matrix has two heads. You will switch between them constantly.

**View 1 — a stack of column vectors.** A matrix is just several vectors lined up side by side. The columns of `A` are `[2, 1]` and `[1, 3]`. This view is what makes matrix multiplication click.

**View 2 — a linear transformation.** A matrix is a *function* that eats vectors and spits out vectors. `A` takes any vector `x` in R² and maps it to a new vector `Ax`, also in R². The function is **linear** — meaning it preserves linear combinations:

$$
A(c_1 \mathbf{x}_1 + c_2 \mathbf{x}_2) = c_1 (A \mathbf{x}_1) + c_2 (A \mathbf{x}_2)
$$

Geometrically, a linear transformation stretches, rotates, reflects, or shears space — but always keeps the origin fixed and keeps grid lines straight and evenly spaced.

These two views are the same thing said two ways. A matrix is the *list of vectors* you get by applying the transformation to the standard basis vectors `e_1, e_2, ..., e_n`. The columns of `A` literally **are** the images of the basis vectors under the transformation. This is the single most useful sentence about matrices, and we will lean on it for the rest of the notebook.


In [ ]:
import numpy as np

A = np.array([[2, 1],
              [1, 3]])

print("A shape   :", A.shape)        # (2, 2)
print("A.T shape :", A.T.shape)      # transpose flips rows/cols
print("column 0  :", A[:, 0])        # [2, 1]
print("column 1  :", A[:, 1])        # [1, 3]
print("row 0     :", A[0, :])        # [2, 1] — yes, same as col 0 here because A is symmetric in shape only


## Matrix-vector multiplication: Ax

The single most important formula in linear algebra. The product `Ax` can be computed two equivalent ways, and you should be fluent in both.

**The row picture.** Each entry of `Ax` is a **dot product** of a row of `A` with `x`:

$$
(A \mathbf{x})_i = \text{(row } i \text{ of } A\text{)} \cdot \mathbf{x}
$$

This is mechanical and easy to compute, but hides the meaning.

**The column picture.** `Ax` is a **linear combination of the columns of `A`**, with the entries of `x` as the weights:

$$
A \mathbf{x} = x_1 \cdot (\text{col }1) + x_2 \cdot (\text{col }2) + \dots + x_n \cdot (\text{col }n)
$$

This is the interpretation that unlocks everything. With `A = [[2, 1], [1, 3]]` and `x = [3, 4]`:

$$
A \mathbf{x} = 3 \begin{bmatrix} 2 \\ 1 \end{bmatrix} + 4 \begin{bmatrix} 1 \\ 3 \end{bmatrix} = \begin{bmatrix} 10 \\ 15 \end{bmatrix}
$$

**Two consequences worth remembering.**

1. The set of all possible outputs `Ax` (over every choice of `x`) is exactly the **span of the columns of `A`** — also called the **column space**. This is a direct restatement of the column picture.
2. Looking at the linear-transformation view: `A · e_j = ` column `j` of `A`. So building a matrix is just "decide where you want the basis vectors to go, then write them as columns."


In [ ]:
x = np.array([3, 4])

# Row picture: dot products
result_row = np.array([A[0] @ x, A[1] @ x])

# Column picture: linear combination of columns
result_col = x[0] * A[:, 0] + x[1] * A[:, 1]

# NumPy's matrix-vector product
result_np = A @ x

print("row picture    :", result_row)
print("column picture :", result_col)
print("A @ x          :", result_np)

# Where do the standard basis vectors land?
print("\nA @ e1 =", A @ np.array([1, 0]), "  <- column 0 of A")
print("A @ e2 =", A @ np.array([0, 1]), "  <- column 1 of A")


## Matrix-matrix multiplication: composition

If `A` is a transformation and `B` is a transformation, then `AB` is the transformation that **first applies `B`, then applies `A`**:

$$
(AB) \mathbf{x} = A (B \mathbf{x})
$$

That ordering — "right one happens first" — trips everyone up at the start. It is the same convention as function composition: `(f ∘ g)(x) = f(g(x))`.

There is a fast way to remember the column rule for `AB`: each **column of `AB`** is `A` applied to the corresponding column of `B`:

$$
\text{col } j \text{ of } AB = A \cdot (\text{col } j \text{ of } B)
$$

So `AB` is just doing the matrix-vector product `A @ b_j` for each column of `B` and stacking the results.

**Order matters: `AB ≠ BA` in general.** Multiplication of numbers is commutative, but composition of transformations is not — rotating then scaling is different from scaling then rotating. When two matrices happen to commute (`AB = BA`), that is special information; it is what makes them simultaneously diagonalizable in the eigenvector notebook.

**Inner dimensions must match.** `A` of shape `(m, k)` times `B` of shape `(k, n)` gives a result of shape `(m, n)`. The `k`'s must match — the number of columns of `A` equals the number of rows of `B` — and they cancel out.


In [ ]:
B = np.array([[0, 1],
              [1, 0]])  # this matrix swaps the two coordinates

print("A @ B =\n", A @ B)
print("B @ A =\n", B @ A)
print("AB == BA?", np.allclose(A @ B, B @ A))

# Verify the "columns of AB" rule
print("\ncol 0 of AB:", (A @ B)[:, 0], "  vs  A @ B[:,0]:", A @ B[:, 0])
print("col 1 of AB:", (A @ B)[:, 1], "  vs  A @ B[:,1]:", A @ B[:, 1])


## Transpose, identity, and inverse

**Transpose** `A^T` flips a matrix across its diagonal — rows become columns, columns become rows. If `A` is `m × n`, then `A^T` is `n × m`. The dot product `u · v` can be written as `u^T v`, treating vectors as one-column matrices. Useful identities:

$$
(AB)^T = B^T A^T, \qquad (A^T)^T = A
$$

The reversal in the first identity is the *only* tricky thing about transposes, but it shows up everywhere — including in the chain rule for backpropagation.

**Identity matrix** `I` is the "do nothing" transformation. It has 1's on the diagonal and 0's everywhere else. `I x = x` for every vector, and `AI = IA = A` for every square matrix. It's the multiplicative identity for matrices.

**Inverse** `A^{-1}` is the matrix that undoes `A`:

$$
A^{-1} A = A A^{-1} = I
$$

Not every matrix has an inverse. A square matrix `A` is **invertible** (also called *non-singular*) if and only if:

- its columns are linearly independent, or equivalently
- its rank equals `n`, or equivalently
- its determinant is non-zero, or equivalently
- the only solution to `Ax = 0` is `x = 0` (trivial null space).

Those four bullets are different phrasings of the same property. Memorize the equivalence — interviewers love asking you to connect them.

Practical warning: even when a matrix is invertible *in theory*, computing `A^{-1}` directly is numerically unstable. In real code, you almost never call `np.linalg.inv(A)`. To solve `Ax = b`, you call `np.linalg.solve(A, b)`. To use `A^{-1}` inside a larger expression, you reformulate to use `solve` instead. We only show `inv` here for clarity.


In [ ]:
# Transpose
print("A   =\n", A)
print("A^T =\n", A.T)

# Identity
I = np.eye(2)
print("\nA @ I = A?", np.allclose(A @ I, A))

# Inverse (for illustration — prefer np.linalg.solve in real code)
A_inv = np.linalg.inv(A)
print("\nA^-1 =\n", A_inv)
print("A @ A^-1 =\n", A @ A_inv)
print("identity? ", np.allclose(A @ A_inv, I))

# Transpose of a product reverses order
B = np.array([[0, 1], [1, 0]])
print("\n(AB)^T == B^T A^T ?", np.allclose((A @ B).T, B.T @ A.T))


## Special matrices

A handful of structured matrices show up so often that they earn names. Each has a property that makes some operation cheap or some guarantee strong.

**Diagonal.** Zero everywhere off the main diagonal. A diagonal matrix `D` acts on `x` by independently scaling each coordinate: `(D x)_i = d_i x_i`. Multiplying, inverting, and exponentiating diagonal matrices is trivial. PCA expresses your data in a basis where the covariance becomes diagonal.

**Symmetric.** `A = A^T`. Equivalently, `A[i, j] = A[j, i]`. Covariance matrices are symmetric. Symmetric matrices have an extremely clean structure: real eigenvalues, orthogonal eigenvectors, always diagonalizable. The **spectral theorem** (notebook 7) is the headline result. Most matrices you optimize over in ML — covariance, Hessian, kernel matrices — are symmetric.

**Orthogonal.** A real square matrix `Q` with `Q^T Q = Q Q^T = I`. Equivalently, the columns of `Q` form an orthonormal basis (each column has length 1, and any two are perpendicular). Orthogonal matrices are exactly the *rigid* linear transformations: they preserve lengths and angles. So `||Q x|| = ||x||` for every `x`. Rotations are orthogonal; reflections are orthogonal. The inverse of an orthogonal matrix is just its transpose — `Q^{-1} = Q^T` — making it the cheapest possible inverse.

Why ML cares:

- **Diagonal** = decoupled features. PCA, batch normalization with diagonal covariance.
- **Symmetric** = nice spectral structure. Covariance, Hessians (in convex problems), kernel methods.
- **Orthogonal** = no distortion. Weight initialization (orthogonal init), invertible neural nets, the `Q` in QR decomposition, rotations in attention.


In [ ]:
# Diagonal
D = np.diag([2, 3])
print("D =\n", D)
print("D @ [1, 1] =", D @ np.array([1, 1]), "  <- scales each coordinate")

# Symmetric
S = np.array([[2, 1], [1, 3]])
print("\nS symmetric? S == S.T:", np.array_equal(S, S.T))

# Orthogonal — a 45-degree rotation
theta = np.pi / 4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
print("\nQ =\n", Q)
print("Q^T Q =\n", Q.T @ Q, "  <- approx identity")
print("||Q @ [3, 4]|| =", np.linalg.norm(Q @ np.array([3, 4])), "  vs  ||[3, 4]|| =", np.linalg.norm([3, 4]))


## Determinant — the volume scaling factor

The **determinant** `det(A)` is a single number that summarizes how a matrix changes volume.

In 2D, picture the unit square — the square with corners at the origin, `e_1`, `e_2`, and `e_1 + e_2`. Apply the transformation `A`. The unit square becomes a **parallelogram** with corners at the origin, `A e_1`, `A e_2`, and `A e_1 + A e_2`. The (signed) area of that parallelogram is `det(A)`.

For our matrix `A = [[2, 1], [1, 3]]`, `det(A) = 2·3 - 1·1 = 5`. So `A` stretches areas by a factor of 5.

Three things the determinant tells you at a glance:

1. **`det(A) ≠ 0` ⇔ `A` is invertible.** A zero determinant means the unit square has been squashed flat — onto a line or a point. Whole dimensions have been collapsed. There is no way to undo that.
2. **`|det(A)|` = the volume scaling factor.** A determinant of 5 multiplies areas (volumes in higher dimensions) by 5. A determinant of 1/2 halves them. A determinant of 1 preserves volume — true for rotations and reflections.
3. **The sign tracks orientation.** Positive determinant preserves orientation; negative flips it (like a reflection in a mirror).

A few useful identities:

$$
\det(AB) = \det(A) \det(B), \qquad \det(A^T) = \det(A), \qquad \det(A^{-1}) = \frac{1}{\det(A)}
$$

In higher dimensions, the same picture works — `det(A)` is the signed `n`-dimensional volume of the image of the unit cube under `A`.


In [ ]:
print("det(A)        =", np.linalg.det(A))     # 5
print("det(I)        =", np.linalg.det(np.eye(2)))
print("det(Q)        =", np.linalg.det(Q))         # 1 (rotation)
print("det(singular) =", np.linalg.det(np.array([[1, 2], [2, 4]])))  # 0 — column 2 is twice column 1


## Where this appears in ML

The matrix is the workhorse of every model. Once you can read `Ax` as "linear combination of columns" and `AB` as "composition of transformations," modern ML is mostly familiar territory:

- **A fully-connected (dense) layer.** `h = activation(W x + b)`. The matrix `W` is a learned linear transformation; the activation adds the only non-linear ingredient.
- **A deep network = composition of layers.** Stacking `n` linear layers is multiplying `n` matrices: `y = W_n W_{n-1} ... W_1 x`. Composition is matrix multiplication.
- **Convolutional layers.** A convolution is a matrix-vector product with a sparse, structured matrix (Toeplitz). The structure (locality, weight sharing) is what makes it cheap and translation-equivariant.
- **Attention.** `attention(Q, K, V) = softmax(Q K^T / √d) V`. Three matrix products, a softmax, one more matrix product. That's it.
- **Backpropagation.** The chain rule, written for matrix-valued functions, becomes `dL/dW = (dL/dh) · x^T`. The transpose comes from `(AB)^T = B^T A^T`.
- **Change of basis.** If columns of `P` are a new basis, then `x' = P^{-1} x` re-expresses `x` in that basis. PCA, whitening, and many preprocessing steps are exactly this.
- **Orthogonal weight initialization.** Initializing weight matrices as orthogonal helps signals propagate without exploding or vanishing — because orthogonal matrices preserve lengths.
- **Determinant in generative models.** Normalizing flows track `log |det(J)|` of the Jacobian to compute exact likelihoods.

Next notebook: **linear systems** — we already met `Ax = b` in the intro. Now we have the vocabulary (column space, rank, null space) to say *exactly* when it has zero, one, or infinitely many solutions, and what each case means for ML.
